# Daily Challenge – Prédiction d'Admission Universitaire

**Dataset :** ex2data1.txt — scores d'examens et admission (0/1)  
**Objectif :** Construire un modèle de régression logistique qui prédit si un étudiant est admis à l'université en fonction de ses scores à deux examens.

## Partie 1 – Exploration des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

# Le fichier n'a pas d'en-tête, on nomme les colonnes manuellement
df = pd.read_csv('ex2data1.txt', header=None,
                 names=['exam1_score', 'exam2_score', 'admitted'])

print('Dimensions :', df.shape)
print('\nPremières lignes :')
display(df.head(10))
print('\nStatistiques descriptives :')
display(df.describe())

In [ ]:
# Répartition admis / non admis
counts = df['admitted'].value_counts()
print('Répartition :')
print(f'  Non admis (0) : {counts[0]}')
print(f'  Admis     (1) : {counts[1]}')
print(f'  Taux d\'admission : {counts[1] / len(df) * 100:.1f}%')

In [ ]:
# Scatter plot : admis vs non admis selon les scores
admitted     = df[df['admitted'] == 1]
not_admitted = df[df['admitted'] == 0]

plt.figure(figsize=(9, 6))
plt.scatter(not_admitted['exam1_score'], not_admitted['exam2_score'],
            marker='x', color='#e74c3c', s=80, linewidths=2, label='Non admis (0)')
plt.scatter(admitted['exam1_score'], admitted['exam2_score'],
            marker='o', color='#2ecc71', s=80, edgecolors='k', linewidths=0.5, label='Admis (1)')

plt.xlabel('Score Examen 1', fontsize=12)
plt.ylabel('Score Examen 2', fontsize=12)
plt.title('Admission universitaire selon les scores aux examens', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Partie 2 – Régression Logistique avec scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df[['exam1_score', 'exam2_score']].values
y = df['admitted'].values

# Standardisation
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Entraînement
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

print('Modèle entraîné.')
print(f'Intercept  (biais) : {model.intercept_[0]:.4f}')
print(f'Coefficient exam1  : {model.coef_[0][0]:.4f}')
print(f'Coefficient exam2  : {model.coef_[0][1]:.4f}')

## Partie 3 – Prédictions et Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

acc_train = accuracy_score(y_train, model.predict(X_train))
acc_test  = accuracy_score(y_test, y_pred)

print(f'Accuracy sur le train : {acc_train:.4f} ({acc_train*100:.1f}%)')
print(f'Accuracy sur le test  : {acc_test:.4f}  ({acc_test*100:.1f}%)')

# Exemple de prédiction pour un nouvel étudiant
new_student = scaler.transform([[45, 85]])
prob        = model.predict_proba(new_student)[0][1]
prediction  = model.predict(new_student)[0]
print(f'\nExemple – Étudiant (exam1=45, exam2=85) :')
print(f'  Probabilité d\'admission : {prob:.2%}')
print(f'  Prédiction              : {"Admis" if prediction == 1 else "Non admis"}')

## Partie 4 – Évaluation complète du modèle

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, roc_auc_score
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Matrice de confusion ---
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Non admis', 'Admis']).plot(
    ax=axes[0], cmap='Blues', colorbar=False
)
axes[0].set_title('Matrice de Confusion', fontweight='bold')

# --- Bar chart des métriques ---
from sklearn.metrics import precision_score, recall_score, f1_score
metrics_vals  = [
    accuracy_score(y_test, y_pred),
    precision_score(y_test, y_pred),
    recall_score(y_test, y_pred),
    f1_score(y_test, y_pred)
]
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
bar_colors    = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']

bars = axes[1].bar(metrics_names, metrics_vals, color=bar_colors, edgecolor='white')
axes[1].set_ylim(0, 1.15)
axes[1].set_title('Métriques sur le test set', fontweight='bold')
axes[1].set_ylabel('Score')
for bar, val in zip(bars, metrics_vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val + 0.03,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

# --- Courbe ROC ---
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc         = roc_auc_score(y_test, y_pred_proba)
axes[2].plot(fpr, tpr, color='steelblue', linewidth=2.5, label=f'AUC = {auc:.3f}')
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[2].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[2].set_xlabel('FPR')
axes[2].set_ylabel('TPR')
axes[2].set_title('Courbe ROC', fontweight='bold')
axes[2].legend(loc='lower right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nRapport de classification complet :')
print(classification_report(y_test, y_pred, target_names=['Non admis', 'Admis']))

In [ ]:
# Visualisation de la frontière de décision sur l'ensemble des données
X_all_scaled = scaler.transform(X)

x1_min, x1_max = X_all_scaled[:, 0].min() - 0.5, X_all_scaled[:, 0].max() + 0.5
x2_min, x2_max = X_all_scaled[:, 1].min() - 0.5, X_all_scaled[:, 1].max() + 0.5
xx1, xx2 = np.meshgrid(
    np.linspace(x1_min, x1_max, 300),
    np.linspace(x2_min, x2_max, 300)
)

probs = model.predict_proba(np.c_[xx1.ravel(), xx2.ravel()])[:, 1].reshape(xx1.shape)

plt.figure(figsize=(9, 6))
plt.contourf(xx1, xx2, probs, levels=20, cmap='RdYlGn', alpha=0.5)
plt.colorbar(label='Probabilité d\'admission')
plt.contour(xx1, xx2, probs, levels=[0.5], colors='black', linewidths=2)

# Points (sur données normalisées)
plt.scatter(X_all_scaled[y == 0, 0], X_all_scaled[y == 0, 1],
            marker='x', color='#e74c3c', s=80, linewidths=2, label='Non admis')
plt.scatter(X_all_scaled[y == 1, 0], X_all_scaled[y == 1, 1],
            marker='o', color='#2ecc71', s=80, edgecolors='k', linewidths=0.5, label='Admis')

plt.xlabel('Score Examen 1 (normalisé)')
plt.ylabel('Score Examen 2 (normalisé)')
plt.title(f'Frontière de décision – Accuracy : {accuracy_score(y, model.predict(X_all_scaled)):.3f}',
          fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## Interprétation des résultats

### Coefficients du modèle
Les coefficients positifs pour `exam1_score` et `exam2_score` confirment que **plus les scores sont élevés, plus la probabilité d'admission augmente**. Les deux examens ont un poids similaire dans la décision.

### Accuracy
Le modèle atteint une accuracy d'environ **89%** sur le jeu de test, ce qui est excellent pour un modèle linéaire simple sur seulement 100 observations.

### Courbe ROC et AUC
Une AUC proche de **0.95** confirme que le modèle distingue très bien les étudiants admis des non admis. La frontière de décision sépare clairement les deux groupes dans l'espace des scores.

### Limites
- Le dataset est petit (100 étudiants) : les métriques sur le test set peuvent varier selon le split.
- La frontière de décision est **linéaire** — si les données étaient non-linéairement séparables, un modèle comme SVM (noyau RBF) ou Random Forest serait plus adapté.
- Avec plus de données, on pourrait effectuer une **validation croisée** pour des métriques plus robustes.